# Lesson 2.2 · Encoding categoricals

Almost all the scikit-learn models accept only numbers. Turning a text column into numbers is a modeling choice,
not a formality: the encoding decides which information (and which false information) the model can use.

**What you will learn**

- To tell apart nominal, ordinal and high-cardinality variables, and choose the right encoding for each.
- To use `OneHotEncoder` and `OrdinalEncoder` with never-seen categories and rare categories.
- To understand why target encoding is powerful and dangerous.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df = ds.loans(with_missing=False)
df[["city", "occupation", "education", "contract_type"]].nunique()

## Three types of categorical variable

| Type | Example | Typical encoding |
|---|---|---|
| **Nominal**, few categories | `contract_type` (4 levels) | one-hot |
| **Ordinal** | `education` (middle school < ... < phd) | ordinal with the order made explicit |
| **High cardinality** | `occupation` (40 levels) | frequency, target encoding, grouping of the rare ones; one-hot only with suitable models and data |

A typical mistake is encoding a nominal variable with integers (Rome = 0, Milan = 1, Naples = 2): the linear model
reads an order and a distance that do not exist.

## One-hot: one column per category

In [ ]:
ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")
M_ohe = ohe.fit_transform(df[["contract_type"]])
print("shape:", M_ohe.shape)
print("columns:", list(ohe.get_feature_names_out()))
pd.DataFrame(M_ohe[:3], columns=ohe.get_feature_names_out(), index=df.index[:3])

`handle_unknown="ignore"` makes a category never seen in training produce a row of zeros instead of an error: it is
almost always what you need in production. `drop="first"` removes one column per category, avoiding redundancy (the
sum of the columns is always 1, that is equal to the intercept): it is needed only with unregularized linear models.

> **Version note.** With `drop` other than `None` scikit-learn 1.1 requires `handle_unknown="error"`. In scikit-learn 1.2 or later the
> parameter `sparse=False` becomes `sparse_output=False`.

## Rare categories: `min_frequency`

In [ ]:
ohe_rare = OneHotEncoder(sparse=False, handle_unknown="infrequent_if_exist", min_frequency=40)
M_rare = ohe_rare.fit_transform(df[["occupation"]])
print("occupation: 40 levels ->", M_rare.shape[1], "columns (the categories with fewer than 40 rows end up together)")
print(list(ohe_rare.get_feature_names_out())[-3:])

## Ordinal: the order must be explicit

In [ ]:
order = ds.EDUCATION_ORDER
oe = OrdinalEncoder(categories=[order], handle_unknown="use_encoded_value", unknown_value=-1)
pd.DataFrame({"education": df.education[:5], "code": oe.fit_transform(df[["education"]])[:5].ravel()})

Without `categories=[order]` the `OrdinalEncoder` sorts the categories alphabetically ("bachelor" < "high school" < "master"
< ...): an order that has nothing to do with the meaning.

## Frequency and target encoding

With 40 occupations a one-hot adds 40 nearly empty columns. Two single-column alternatives:

- **Frequency encoding**: every category becomes its frequency. Simple and with no leakage, but it loses the information about the target.
- **Target encoding**: every category becomes the mean (with *smoothing*) of the target in the category. Very informative and very
  dangerous: if the mean of a row is computed **including the row itself**, the encoding contains the target. It is
  computed out-of-fold, as you will see in exercises A3 and E3.

In [ ]:
freq = df["occupation"].map(df["occupation"].value_counts(normalize=True))
te_naive = df["occupation"].map(df.groupby("occupation")["defaulted"].mean())     # it includes the row itself!
print("occupations with fewer than 10 rows:", int((df["occupation"].value_counts() < 10).sum()))
print("correlation of the 'naive' target encoding with the target:", round(np.corrcoef(te_naive, df["defaulted"])[0, 1], 3))

> **Common mistake.** Encoding a nominal variable with integers (1, 2, 3...) and giving it to a linear model or to a KNN: the model reads an order and a distance that do not exist. Integers are fine only for ordinals with the order declared, or for trees (exercise C2). And building the dummies with `pd.get_dummies` separately on training and test produces different columns (exercise E1).

## Check your understanding

<details><summary>Why don't I encode `city` with numbers from 0 to 11?</summary>

Because a linear model (or KNN) would treat city 11 as 'eleven times city 1' and city 5 as 'halfway between 0 and 10': an invented order and distance. The correct encodings for a nominal are one-hot, frequency or target encoding. A tree can get by with numbers (with enough splits it isolates every category), but it is an exception.

</details>

<details><summary>What happens to `OneHotEncoder(handle_unknown='error')` in production if a new city arrives?</summary>

It raises an error and the service stops. With `handle_unknown='ignore'` the row of zeros is a prudent choice: the model sees 'none of the known cities'. In both cases it is worth monitoring the appearance of new categories: it is a drift signal.

</details>